<a href="https://colab.research.google.com/github/ZaidKhan2002/GenAI_Experimentation/blob/main/chat_loa_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit  # optional (only needed if you plan to use the original version)
!pip install langchain-community
!pip install sentence-transformers
!pip install faiss-cpu
!pip install cohere
!pip install PyPDF2
!pip install numpy
!pip install pandas
!pip install pypdf



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

In [ ]:
import os
import getpass

os.environ["COHERE_API_KEY"] = getpass.getpass("Enter Cohere API Key: ")

Enter Cohere API Key: ··········


In [ ]:
import numpy as np
import pandas as pd
import faiss
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from cohere import ClientV2

# Initialize Sentence Transformer model
model_name = "all-MiniLM-L6-v2"  # Adjust model as needed
sentence_transformer_model = SentenceTransformer(model_name)

# Load and process the PDF file
pdf_path = 'Rhea_resume.pdf'
loader = PyPDFLoader(pdf_path)
documents = loader.load()

embeddings = []
documents_text = []
sources = []

# To chunkify the docs
latex_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

for docu in documents:  # one page at a time from the PDF
    docs = latex_splitter.create_documents([docu.page_content])  # splitting into chunks
    for document in docs:
        document_embedding = sentence_transformer_model.encode(document.page_content)
        embeddings.append(document_embedding)
        documents_text.append(document.page_content)
        sources.append("www.rheadata.com")

# Create FAISS index
embedding_dimension = len(embeddings[0])
index = faiss.IndexFlatL2(embedding_dimension)
index.add(np.array(embeddings, dtype='float32'))

# Save index and document details
if not os.path.exists("Vector_Store"):
    os.makedirs("Vector_Store")

df = pd.DataFrame({'documents': documents_text, 'source': sources})
df.to_csv('Vector_Store/docs.csv', index=False)
faiss.write_index(index, 'Vector_Store/vector_db.index')

print("✅ FAISS index and document store created successfully!")


/tmp/ipykernel_777/1262708446.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ FAISS index and document store created successfully!


In [ ]:
embeddings

[array([-7.44969323e-02,  1.85878258e-02, -3.14883105e-02, -1.21733777e-01,
        -5.10463826e-02, -8.12959746e-02,  1.63074937e-02, -1.25655169e-02,
        -3.49160396e-02, -2.57054940e-02, -3.66459563e-02, -2.31535286e-02,
         9.20505524e-02, -6.38431162e-02, -6.97143525e-02,  6.31419793e-02,
         7.63854431e-03, -8.63823369e-02,  3.62240709e-02, -8.99307653e-02,
         2.06122808e-02, -1.24284727e-02, -4.74636815e-02, -4.03432623e-02,
         5.59965521e-02,  4.21531126e-02,  5.21758310e-02, -1.24118605e-03,
        -1.20925736e-02,  1.12930349e-04, -3.38876364e-03,  7.72572216e-03,
         2.00740583e-02,  2.79349014e-02, -1.65899433e-02,  7.28938878e-02,
         1.38736200e-02, -4.94905189e-03, -4.37941849e-02,  4.41368781e-02,
        -9.99859869e-02,  2.44252495e-02,  1.53435497e-02,  4.90401797e-02,
         3.93509679e-02, -6.47781491e-02, -4.53308560e-02, -5.39937392e-02,
         9.58978012e-03, -3.27259265e-02, -5.52182905e-02,  4.95329360e-03,
        -1.7

In [ ]:

# --- Q&A Section ---

# Load the FAISS index and CSV (optional reload)
index = faiss.read_index('Vector_Store/vector_db.index')
df = pd.read_csv('Vector_Store/docs.csv')

# Initialize Cohere client
co = ClientV2(api_key=os.environ["COHERE_API_KEY"])

# User query input
query = input("Enter your query: ")

if query:
    query_embedding = sentence_transformer_model.encode(query).reshape(1, -1)
    distances, indices = index.search(query_embedding, k=5)

    threshold = 1.7  # Define threshold for distance match
    print("Distance score:", distances)

    if distances[0][0] > threshold:
        print(" Please ask a relevant question.")
    else:
        combined_similar_documents_content_list = []
        similar_documents_sources = []

        for i in indices[0]:
            similar_document_content = df.loc[i, 'documents']
            combined_similar_documents_content_list.append(similar_document_content)

            similar_document_source = df.loc[i, 'source']
            similar_documents_sources.append(similar_document_source)

        combined_similar_documents_content = ' '.join(combined_similar_documents_content_list)

        cohere_prompt = (
            f"Based on the document content: {combined_similar_documents_content}, "
            f"answer the question: '{query}'"
        )

        # Call Cohere API
        cohere_response = co.chat(
            model="command-a-03-2025",
            messages=[{"role": "user", "content": cohere_prompt}],
            temperature=0.3
        )

        print("\n Bot Response:")
        print(cohere_response.message.content[0].text)

        print("\n Sources:")
        print(list(set(similar_documents_sources)))


Enter your query: Experiences in designing?
Distance score: [[1.5369048 1.5553718 1.6256094 1.685916  1.7873425]]

 Bot Response:
Based on the document, the candidate has several experiences in designing, particularly in **UI/UX design** and **project development**. Here are the key experiences:

1. **NutriCampus Project (Sept. 2024)**  
   - **Designed UI/UX using HTML and CSS** for a React.js platform that offers personalized meal recommendations from campus dining menus.  
   - This demonstrates hands-on experience in front-end design and user experience optimization.

2. **General Engineering and Computer Science Education (Aug. 2024 – Present)**  
   - Currently enrolled in courses like **Software Design and Data Structures**, which involve designing software systems and user interfaces.  
   - This academic experience provides a foundational understanding of design principles in software development.

3 **Tim Duong Research Laboratory (June 2024 – Present)**  
   - While the focu